# 05 — The Zeta Index: Spectral Wavelengths and Prime Resolution

**The Zero Tree / Telperion — Notebook 5**

---

## The Foundational Statement

Each non-trivial Riemann zero ρₙ = ½ + iγₙ has a **spectral wavelength in logarithmic space**:

$$\lambda_n = \frac{2\pi}{\gamma_n}$$

The zero ρₙ operates at linear scale:

$$L_n(x) = \frac{2\pi x}{\gamma_n} \quad \text{(resolution at scale } x\text{)}$$

It completes its first oscillation cycle — **activates** — at:

$$x_n = e^{2\pi/\gamma_n}$$

This follows from the explicit formula for π(x):
$$\pi(x) = \text{li}(x) - \sum_{\rho} \text{li}(x^\rho) - \log 2 + \int_x^\infty \frac{dt}{t(t^2-1)\log t}$$

The contribution of zero ρₙ oscillates as $x^{1/2} \cos(\gamma_n \log x)$ —
with period $2\pi/\gamma_n$ in log-space, and linear wavelength $2\pi x/\gamma_n$ near x.

---

## The Zeta Index Definition

By the prime number theorem, the average prime gap near p is log(p).
Zero ρₙ **resolves** prime p when its linear wavelength at p is ≤ the local gap:

$$L_n(p) \leq \log(p) \implies \frac{2\pi p}{\gamma_n} \leq \log(p) \implies \gamma_n \geq \underbrace{\frac{2\pi p}{\log p}}_{\gamma^*(p)}$$

**The zeta index of prime p:**

$$\boxed{\zeta(p) = \min\left\{\, n \in \mathbb{N} \;:\; \gamma_n \geq \frac{2\pi p}{\log p} \,\right\}}$$

ζ(p) is the index of the first Riemann zero whose resolution is fine enough to distinguish consecutive primes near p.

## The Double Index

Each prime now carries two independent indices:

| Index | Symbol | Meaning |
|---|---|---|
| Ordinal | n = π(p) | p is the n-th prime (position in ℕ ordering) |
| Zeta | ζ(p) | first zero to resolve p (spectral emergence index) |

Canonical notation: **p_{n[ζ(p)]}** — ordinal n, zeta sub-index ζ(p).

Example: p=17 (17th prime... no, 7th prime) → **p_{7[7]}** — the 7th prime, resolved by the 7th zero.

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join('..', 'engine'))

# FermatMonster dependency (for prime sieve)
fermat_path = os.path.join('..', '..', '..', 'FourthAgePapers', 'FermatMonster', 'engine')
if os.path.exists(fermat_path):
    sys.path.insert(0, fermat_path)

from zeta_index_engine import (
    ZetaIndexEngine, spectral_wavelength, activation_scale, gamma_threshold, zeta_index,
    KNOWN_ZEROS_100, TWO_PI
)
from telperion_engine import prime_sieve

# Use precomputed zeros for reproducibility (no mpmath dependency in notebook)
engine = ZetaIndexEngine(n_zeros=100)
engine._zeros = list(KNOWN_ZEROS_100)  # use precomputed table
primes = prime_sieve(200)

print(f'Primes loaded: {len(primes)} primes ≤ 200')
print(f'Zeros loaded: {len(KNOWN_ZEROS_100)} precomputed Riemann zeros')
print(f'γ₁  = {KNOWN_ZEROS_100[0]:.6f}  λ₁ = {spectral_wavelength(KNOWN_ZEROS_100[0]):.6f}  x₁ = {activation_scale(KNOWN_ZEROS_100[0]):.6f}')
print(f'γ₁₀ = {KNOWN_ZEROS_100[9]:.6f}  λ₁₀ = {spectral_wavelength(KNOWN_ZEROS_100[9]):.6f}  x₁₀ = {activation_scale(KNOWN_ZEROS_100[9]):.6f}')

## Part 1 — The Zeros and Their Wavelengths

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

zeros = KNOWN_ZEROS_100
ns    = list(range(1, len(zeros)+1))
lams  = [spectral_wavelength(g) for g in zeros]
xs    = [activation_scale(g)    for g in zeros]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# γₙ vs n
axes[0].plot(ns, zeros, 'o-', color='#334466', markersize=3, linewidth=0.8)
axes[0].set_xlabel('n')
axes[0].set_ylabel('γₙ')
axes[0].set_title('Riemann zeros γₙ vs index n')

# λₙ = 2π/γₙ vs n  (log scale)
axes[1].semilogy(ns, lams, 'o-', color='#664433', markersize=3, linewidth=0.8)
axes[1].set_xlabel('n')
axes[1].set_ylabel('λₙ = 2π/γₙ  (log scale)')
axes[1].set_title('Spectral wavelength λₙ in log-space')

# Activation scale x_n = e^λₙ vs n
axes[2].plot(ns, xs, 'o-', color='#226633', markersize=3, linewidth=0.8)
axes[2].axhline(1, color='black', linewidth=0.5, linestyle='--')
axes[2].set_xlabel('n')
axes[2].set_ylabel('x_n = e^(2π/γₙ)')
axes[2].set_title('Activation scale: first cycle completes at x_n')
axes[2].set_ylim(1, 2)

plt.suptitle('Riemann zeros: spectral wavelengths and activation scales (n=1..100)', y=1.01)
plt.tight_layout()
plt.savefig('05_zero_wavelengths.png', dpi=150)
plt.show()
print('\nNote: all activation scales x_n ≈ 1.1–1.6 — zeros activate very early')
print('The RESOLUTION (distinguishing primes) requires γₙ ≥ 2πp/log(p)')

## Part 2 — The Zeta Index: Which Zero First Resolved Each Prime?

In [ ]:
NIEMEIER_GAP = {1, 11, 15}

records = engine.prime_zeta_table(primes)
# Filter to primes where ζ is defined (within table)
valid   = [r for r in records if r['zeta_idx'] > 0]

print(f'Primes with ζ(p) within 100-zero table: {len(valid)} of {len(primes)}')
print()
print(f'  {"p":>5}  {"n":>4}  {"e_k":>4}  {"γ*(p)":>10}  {"ζ(p)":>5}  {"γ_ζ":>10}  {"Double index":>15}')
print('  ' + '-' * 65)
for r in valid[:20]:
    mg  = '★' if r['nshape'] in NIEMEIER_GAP else ' '
    di  = f'p_{{{r["n"]}[{r["zeta_idx"]}]}}'
    print(f'  {mg}{r["p"]:>4}  {r["n"]:4d}  e{r["nshape"]:2d}  '
          f'{r["gamma_star"]:10.3f}  {r["zeta_idx"]:5d}  '
          f'{r["gamma_zeta"]:10.3f}  {di:>15}')

In [ ]:
# Scatter: ordinal n vs zeta index ζ(p), colored by N-shape
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Collect data
n_vals  = [r['n']       for r in valid]
z_vals  = [r['zeta_idx'] for r in valid]
ns_vals = [r['nshape']  for r in valid]
colors  = ['silver' if ns in NIEMEIER_GAP else '#334466' for ns in ns_vals]

axes[0].scatter(n_vals, z_vals, c=colors, s=20, alpha=0.8, zorder=3)
axes[0].plot(n_vals, z_vals, '-', color='#aaaaaa', linewidth=0.3, alpha=0.5)
axes[0].set_xlabel('n (ordinal index)')
axes[0].set_ylabel('ζ(p) (zeta index)')
axes[0].set_title('Double index: ordinal n vs zeta index ζ(p)\n(silver = Monster gap primes)')

# γ*(p) vs p — the threshold curve
p_range   = list(range(2, max(r['p'] for r in valid)+1))
thresh    = [gamma_threshold(p) for p in p_range]
axes[1].plot(p_range, thresh, color='#334466', linewidth=1.5, label='γ*(p) = 2πp/log(p)')

# Mark the zeros as horizontal lines
for zi in range(1, 26):
    g = KNOWN_ZEROS_100[zi-1]
    axes[1].axhline(g, color='#cc8833', linewidth=0.4, alpha=0.6)
    if zi <= 10:
        axes[1].text(max(p_range)*1.01, g, f'γ_{zi}', fontsize=6, va='center')

axes[1].set_xlabel('p (prime)')
axes[1].set_ylabel('γ')
axes[1].set_title('Threshold γ*(p) = 2πp/log(p) vs Riemann zeros (horizontal lines)\nZero ρₙ resolves p when γₙ ≥ γ*(p)')
axes[1].legend()

plt.tight_layout()
plt.savefig('05_zeta_index.png', dpi=150)
plt.show()

## Part 3 — Spectral Sub-Ordering

In [ ]:
# The zeta ordering: primes sorted by spectral emergence
ordering = engine.zeta_ordering(primes)
valid_ord = [(zi, n, p) for zi, n, p in ordering if zi > 0]

print('Zeta sub-ordering — primes by spectral emergence:')
print('(which prime the Riemann spectrum resolved FIRST)')
print()
print(f'  {"Rank":>5}  {"ζ(p)":>5}  {"n":>4}  {"p":>5}  {"e_k":>4}  Double index')
print('  ' + '-' * 50)
for rank, (zi, n, p) in enumerate(valid_ord[:30], 1):
    mg  = '★' if p%16 in NIEMEIER_GAP else ' '
    di  = f'p_{{{n}[{zi}]}}'
    print(f'  {rank:5d}  {zi:5d}  {n:4d}  {mg}{p:4d}  e{p%16:2d}  {di}')

In [ ]:
# Compare Monster gap vs other primes in zeta distribution
mg_data    = engine.monster_gap_zeta(primes)
gap_valid  = [z for z in mg_data['gap_zetas']   if z > 0]
other_valid= [z for z in mg_data['other_zetas'] if z > 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = range(1, max(max(gap_valid, default=1), max(other_valid, default=1)) + 2)
axes[0].hist(gap_valid,   bins=bins, alpha=0.7, color='silver', label='Monster gap (e₁,e₁₁,e₁₅)', edgecolor='gray')
axes[0].hist(other_valid, bins=bins, alpha=0.5, color='#334466', label='Other N-shapes', edgecolor='white')
axes[0].axvline(sum(gap_valid)/len(gap_valid)   if gap_valid   else 0, color='gray',    linestyle='--', linewidth=1.5, label=f'Gap mean = {sum(gap_valid)/len(gap_valid):.1f}'   if gap_valid   else '')
axes[0].axvline(sum(other_valid)/len(other_valid) if other_valid else 0, color='#334466', linestyle='--', linewidth=1.5, label=f'Other mean = {sum(other_valid)/len(other_valid):.1f}' if other_valid else '')
axes[0].set_xlabel('ζ(p)')
axes[0].set_ylabel('Count')
axes[0].set_title('ζ(p) distribution: Monster gap vs other primes')
axes[0].legend(fontsize=8)

# N-shape × zeta heatmap
ns_zeta = {}
for r in valid:
    ns = r['nshape']
    if ns not in ns_zeta:
        ns_zeta[ns] = []
    ns_zeta[ns].append(r['zeta_idx'])

ns_keys  = sorted(ns_zeta.keys())
ns_means = [sum(ns_zeta[ns])/len(ns_zeta[ns]) for ns in ns_keys]
bar_cols = ['silver' if ns in NIEMEIER_GAP else '#334466' for ns in ns_keys]
axes[1].bar(ns_keys, ns_means, color=bar_cols, edgecolor='white')
axes[1].set_xticks(ns_keys)
axes[1].set_xticklabels([f'e{ns}' for ns in ns_keys], rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Mean ζ(p)')
axes[1].set_title('Mean zeta index by N-shape (silver = Monster gap)')

plt.tight_layout()
plt.savefig('05_monster_gap_zeta.png', dpi=150)
plt.show()

print(f'Monster gap mean ζ:  {sum(gap_valid)/len(gap_valid):.2f}  ({len(gap_valid)} primes)')
print(f'Other primes mean ζ: {sum(other_valid)/len(other_valid):.2f}  ({len(other_valid)} primes)')

## Part 4 — Connection to the Hyperwebster Address Chain

In [ ]:
# The Hyperwebster chain extended with zeta index:
#
#  word
#    → Horner hash H(w) = Σ ord(cᵢ) × 95^(|w|-i)
#    → prime p = next_prime(H(w) mod 2^16)
#    → ordinal n = π(p)         ← word's ordinal address on σ=½
#    → γₙ                       ← zero at that index: word's energy
#    → E = |sin(π × γₙ / (γₙ+1))|  ← field energy
#    NEW:
#    → γ*(p) = 2πp/log(p)       ← spectral resolution threshold
#    → ζ(p)                     ← zeta index: first zero to resolve p
#    → (n, ζ(p))                ← double index on this word's prime
#
# The double index (n, ζ(p)) adds spectral depth to the address:
#   n = WHERE on the critical line the word lives (its Riemann zero)
#   ζ(p) = WHEN the Riemann spectrum first resolved this word's prime

# Demo: trace a few words through the extended chain
import math

def horner_hash(word, base=95, offset=32):
    v = 0
    for c in word:
        v = v * base + (ord(c) - offset)
    return v

def next_prime(n, prime_list):
    n = max(2, n % 65536)
    for p in prime_list:
        if p >= n:
            return p
    return prime_list[-1]

ALL_PRIMES = prime_sieve(70000)

words = ['zero', 'prime', 'telperion', 'monster', 'piano', 'melancholy', 'persists', 'silver']
print(f'  {"word":<15}  {"p":>6}  {"n=π(p)":>7}  {"γₙ":>10}  {"γ*(p)":>10}  {"ζ(p)":>6}  {"double_idx"}')
print('  ' + '-' * 80)

for word in words:
    h  = horner_hash(word)
    p  = next_prime(h, ALL_PRIMES)
    # ordinal = π(p): count primes ≤ p
    n  = sum(1 for q in ALL_PRIMES if q <= p)
    # zero at ordinal n
    zi = engine.zeta_index(p)
    g_n = KNOWN_ZEROS_100[min(n-1, len(KNOWN_ZEROS_100)-1)] if n <= len(KNOWN_ZEROS_100) else None
    g_star = gamma_threshold(p)
    di = f'p_{{{n}[{zi}]}}' if zi > 0 else f'p_{{{n}[>100]}}'
    g_str = f'{g_n:.4f}' if g_n else '—'
    mg = ' ★' if p%16 in NIEMEIER_GAP else ''
    print(f'  {word:<15}  {p:6d}  {n:7d}  {g_str:>10}  {g_star:10.3f}  {zi:6d}  {di}{mg}')

## Summary

```
SPECTRAL WAVELENGTH THEOREM (canonical)

Zero ρₙ = ½ + iγₙ has log-space wavelength:
    λₙ = 2π/γₙ

Linear resolution at scale x:
    Lₙ(x) = 2πx/γₙ

Activation scale (first cycle):
    x_n = e^(2π/γₙ)

Resolution threshold for prime p (average gap = log p by PNT):
    γ*(p) = 2πp/log(p)

Zeta index (first zero to resolve p):
    ζ(p) = min{ n : γₙ ≥ 2πp/log(p) }

Double index on the leaf pₙ of Telperion:
    p_{n[ζ(p)]}   — ordinal n, zeta sub-index ζ(p)

Monotonicity:  p₁ ≤ p₂  ⟹  ζ(p₁) ≤ ζ(p₂)
(γ*(p) = 2πp/log(p) is increasing for p ≥ e)

The double index (n, ζ(p)) gives Telperion's leaves
a two-dimensional spectral address:
  n     = WHERE the prime lives on σ=½
  ζ(p)  = WHEN the Riemann spectrum first resolved it
```